In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv(path + '/Q3_data.csv')

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
#for some reason it doesn't show th info, i think because th enumber of columns
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
print(df.isna().sum().sum())
print(df.isna().sum())

for col in df.select_dtypes(exclude= 'object').columns:
  df[col] = df[col].fillna(df[col].mean().astype(df[col].dtype))

for col in df.select_dtypes(include= 'object').columns:
  df[col] = df[col].fillna(df['Weather'].mode().values[0])
print('after -- \n')
print(df.isna().sum().sum())

print(df.isna().sum())



In [ ]:
# Task 2: Write your code here:
print(df.duplicated().sum())
# there is no duplicates

In [ ]:
# Task 3: Write your code here:
df.select_dtypes(include= 'object').columns

#we don't have any objects so no need

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
#split to target and features
X = df.drop('Target', axis= 1)
y = df['Target']

s_scaler = StandardScaler()
X_scaled = s_scaler.fit_transform(X)
print(type(X_scaled))
#convert to pandas
X_scaled = pd.DataFrame(X_scaled, columns=s_scaler.get_feature_names_out(), index= X.index)
X_scaled.head()



In [ ]:
# Task 5: Write your code here:
df['Target'].value_counts(normalize= True)

# we do have class imbalanaced

In [ ]:
# Task 1: Write your code here:
#i already did this
X = X_scaled
y = y

In [ ]:
# Task 2,3,4,5: Write your code here:
# as we have imbalance so we need stratify
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score

#random state to make it always li,e this distr
skf = StratifiedKFold(5, random_state= 42, shuffle= True)

all_loss = 0
num_of_folds = 0

cbc = CatBoostClassifier(verbose=0, n_estimators=200, max_depth=4)

for train_index, test_index in skf.split(X, y):
    X_train, y_train = X.iloc[train_index, :], y.iloc[train_index]
    X_test, y_test = X.iloc[test_index, :], y.iloc[test_index]


    cbc.fit(X_train, y_train)
    preds = cbc.predict(X_test)

    #as we have imbalance we will use f1 (i used it in my studing session)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)

    all_loss += f1
    num_of_folds +=1

print(f"avg = {all_loss/num_of_folds}")




In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
#last model iteration's will be used to get feature importance
import matplotlib.pyplot as plt
importance = pd.Series(cbc.feature_importances_, index = X.columns )
#sort it
importance.sort_values(ascending= False, inplace= True)
#plot feature importances
plt.figure(figsize=(30, 30))
plt.barh(importance.index, importance.values,  color='red')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('last model importance')
plt.show()



In [ ]:
# Task 2: Write your code here:
# as i sorted it i can just use it like this
print(f'the most important feature is: {importance.index[0]} with {importance.iloc[0]}')

In [ ]:
cbc.predict?

In [ ]:
CatBoostClassifier?

In [ ]:
# Task Bonus: Write your code here:
X = X_scaled['P_2']

#random state to make it always li,e this distr
skf = StratifiedKFold(5, random_state= 42, shuffle= True)

all_loss = 0
num_of_folds = 0

cbc = CatBoostClassifier(verbose=0, n_estimators=200, max_depth=4)

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    cbc.fit(X_train.to_numpy().reshape(-1,1), y_train)
    preds = cbc.predict(X_test.to_numpy().reshape(-1,1))
    #as we have imbalance we will use f1 (i used it in my studing session)
    f1 = f1_score(y_test, preds, average='macro')

    all_loss += f1
    num_of_folds +=1

print(f"avg = {all_loss/num_of_folds}")


